## Requisitos y limitaciones

- Requiere las dependencias opcionales de aprendizaje profundo; TensorFlow puede necesitar mucha memoria y tiempo. Los ejemplos con URLs/modelos descargan recursos de Internet y algunos usan API antiguas, por lo que pueden requerir adaptación de compatibilidad según la versión instalada. No se instala TensorFlow en la imagen base.
- Este ejemplo puede acceder a datos externos en tiempo de ejecución; requiere conectividad de red.

- Usa `tflearn`, API heredada sin extra compatible incluido. Puede no ser compatible con las versiones actuales de TensorFlow/Keras; el resto de notebooks TensorFlow no depende de tflearn.

In [ ]:
# -*- coding: utf-8 -*-
# pip install h5py para rendimiento
#Ojito qu etarda un montón en ejecutar

""" Convolutional network applied to CIFAR-10 dataset classification task.
References:
    Learning Multiple Layers of Features from Tiny Images, A. Krizhevsky, 2009.
Links:
    [CIFAR-10 Dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
"""
from __future__ import division, print_function, absolute_import

import tflearn
from tflearn.data_utils import shuffle, to_categorical
from tflearn.layers.core import input_data, dropout, fully_connected
from tflearn.layers.conv import conv_2d, max_pool_2d
from tflearn.layers.estimator import regression
from tflearn.data_preprocessing import ImagePreprocessing
from tflearn.data_augmentation import ImageAugmentation
import tensorflow as tf

# Data loading and preprocessing
from keras.datasets import cifar10
(X, Y), (X_test, Y_test) = cifar10.load_data()
X, Y = shuffle(X, Y)
Y = to_categorical(Y, nb_classes=10)
Y_test = to_categorical(Y_test, nb_classes=10)

# Real-time data preprocessing
img_prep = ImagePreprocessing()
img_prep.add_featurewise_zero_center()
img_prep.add_featurewise_stdnorm()

# Real-time data augmentation
img_aug = ImageAugmentation()
img_aug.add_random_flip_leftright()
img_aug.add_random_rotation(max_angle=25.)

# Convolutional network building
network = input_data(shape=[None, 32, 32, 3],
                     data_preprocessing=img_prep,
                     data_augmentation=img_aug)
network = conv_2d(network, 32, 3, activation='relu')
network = max_pool_2d(network, 2)
network = conv_2d(network, 64, 3, activation='relu')
network = conv_2d(network, 64, 3, activation='relu')
network = max_pool_2d(network, 2)
network = fully_connected(network, 512, activation='relu')
network = dropout(network, 0.5)
network = fully_connected(network, 10, activation='softmax')
network = regression(network, optimizer='adam',
                     loss='categorical_crossentropy',
                     learning_rate=0.001)
logs_path = 'logs/image2'
# Train using classifier
# tensorboard --logdir='logs/image2'
model = tflearn.DNN(network, tensorboard_verbose=1, tensorboard_dir=logs_path)
model.fit(X, Y, n_epoch=50, shuffle=True, validation_set=(X_test, Y_test),
          show_metric=True, batch_size=5000, run_id='cifar10_cnn')
score = model.evaluate(X_test, Y_test)
print("Score", score)

